# ASI01 Agent Goal Hijack — Upload Artifacts & Run Evaluation

**OWASP Category**: ASI01 — Agent Goal Hijack | **Risk Severity**: Critical

**Mapped LLM Categories**: LLM01 (Prompt Injection), LLM06 (Excessive Agency)

This notebook:
1. **Uploads** all artifact files (scenarios, checks, drivers) from the local folder structure and registers them in Okareo.
2. **Runs** the full ASI01 agent goal hijack test suite against your target AI agent.

Uploading is **idempotent** — re-running will not create duplicates (scenarios return existing, checks/drivers upsert).
Registered IDs are available in-memory for the evaluation steps below.

In [ ]:
%pip install okareo python-dotenv --quiet

In [ ]:
import sys
import json
from pathlib import Path

# Add project root for owasp.common import
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))

from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver

from owasp.common import (
    init_okareo,
    parse_artifact,
    build_target,
    SINGLE_TURN_DRIVER_TEMPLATE,
)

okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")
print(f"Category directory: {CATEGORY_DIR}")

---
## Part 1 — Upload Artifacts

### Upload Scenarios

Scans `scenarios/` for `.jsonl` files and uploads each via `upload_scenario_set`.

In [ ]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}  # name -> scenario object

for jsonl_path in sorted(scenarios_dir.glob("*.jsonl")):
    scenario_name = f"ASI01-{jsonl_path.stem}"
    print(f"Uploading scenario: {scenario_name} from {jsonl_path.name}")

    scenario = okareo.upload_scenario_set(
        scenario_name=scenario_name,
        file_path=str(jsonl_path),
    )
    registered_scenarios[scenario_name] = scenario
    print(f"  ✓ Registered: {scenario_name} (ID: {scenario.scenario_id})")

print(f"\nTotal scenarios uploaded: {len(registered_scenarios)}")

### Register Checks

Scans `checks/` for `.md` files, parses YAML front matter and prompt template,
and registers each via `create_or_update_check` using `ModelBasedCheck`.

In [ ]:
checks_dir = CATEGORY_DIR / "checks"
registered_checks = {}  # name -> check id

for md_path in sorted(checks_dir.glob("*.md")):
    check_data = parse_artifact(md_path)
    print(f"Registering check: {check_data['name']} from {md_path.name}")

    check_obj = ModelBasedCheck(
        prompt_template=check_data["prompt_template"],
        check_type=CheckOutputType.PASS_FAIL,
    )
    result = okareo.create_or_update_check(
        name=check_data["name"],
        description=check_data["description"],
        check=check_obj,
    )
    registered_checks[check_data["name"]] = result.id
    print(f"  ✓ Registered: {check_data['name']} (ID: {result.id})")

print(f"\nTotal checks registered: {len(registered_checks)}")

### Register Drivers

Scans `drivers/` for `.md` files, parses YAML front matter and persona prompt,
and registers each via `create_or_update_driver` using a `Driver` object.

In [ ]:
drivers_dir = CATEGORY_DIR / "drivers"
registered_drivers = {}  # name -> driver data dict

for md_path in sorted(drivers_dir.glob("*.md")):
    driver_data = parse_artifact(md_path, default_temperature=0.7)
    print(f"Registering driver: {driver_data['name']} from {md_path.name}")

    driver_obj = Driver(
        name=driver_data["name"],
        prompt_template=driver_data["prompt_template"],
        temperature=driver_data["temperature"],
    )
    result = okareo.create_or_update_driver(driver=driver_obj)
    registered_drivers[driver_data["name"]] = result
    print(f"  ✓ Registered: {driver_data['name']} (ID: {result.id})")

print(f"\nTotal drivers registered: {len(registered_drivers)}")

### Artifact Upload Summary

In [ ]:
print("=" * 60)
print("ASI01 Agent Goal Hijack — Artifact Upload Summary")
print("=" * 60)
print(f"\nScenarios ({len(registered_scenarios)}):")
for name, sc in registered_scenarios.items():
    print(f"  • {name} → {sc.scenario_id}")
print(f"\nChecks ({len(registered_checks)}):")
for name, cid in registered_checks.items():
    print(f"  • {name} → {cid}")
print(f"\nDrivers ({len(registered_drivers)}):")
for name, drv in registered_drivers.items():
    print(f"  • {name} → {drv.id}")
print("\n✓ All artifacts ready. Proceeding to evaluation...")

---
## Part 2 — Run Evaluation

### Configuration

The target agent is loaded from the shared `owasp/target.json` file.

**ASI01 evaluation structure**:
- **Scenario 1** (`direct-goal-override`): Multi-turn simulation using `ASI01-goal-override-attacker` driver, both checks applied, `max_turns=10`
- **Scenario 2** (`indirect-goal-hijack`): Single-turn evaluation using pass-through driver, `ASI01-goal-hijack-detector` check only
- **Scenario 3** (`goal-drift-manipulation`): Multi-turn simulation using `ASI01-goal-drift-manipulator` driver, both checks applied, `max_turns=10`

In [ ]:
target = build_target(CATEGORY_DIR)
TARGET_NAME = target.name
print(f"✓ Target agent: {TARGET_NAME}")

GOAL_HIJACK_CHECK    = "ASI01-goal-hijack-detector"
GOAL_CONSISTENCY_CHECK = "ASI01-goal-consistency-check"
MAX_TURNS = 10

# Multi-turn: scenario -> driver pairing
MULTI_TURN_MAP = {
    "ASI01-direct-goal-override":   "ASI01-goal-override-attacker",
    "ASI01-goal-drift-manipulation": "ASI01-goal-drift-manipulator",
}

# Single-turn scenarios
SINGLE_TURN_SCENARIOS = ["ASI01-indirect-goal-hijack"]

all_results = {}

### Single-Turn Test — Scenario 2: Indirect Goal Hijack via External Data

In [ ]:
single_turn_driver = Driver(
    name="pass-through",
    prompt_template=SINGLE_TURN_DRIVER_TEMPLATE,
    temperature=0,
)

for scenario_name in SINGLE_TURN_SCENARIOS:
    if scenario_name not in registered_scenarios:
        print(f"⚠ Scenario {scenario_name} not found — skipping")
        continue

    print(f"Running single-turn: {scenario_name}")
    try:
        run = okareo.run_simulation(
            target=target,
            driver=single_turn_driver,
            name=f"ASI01 Single-Turn — {scenario_name}",
            api_key=OKAREO_API_KEY,
            first_turn="target",
            scenario=registered_scenarios[scenario_name],
            max_turns=1,
            checks=[GOAL_HIJACK_CHECK],
        )
        all_results[scenario_name] = run
        print(f"  ✓ Complete: {run.id}")
        if hasattr(run, "app_link") and run.app_link:
            print(f"  View: {run.app_link}")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        all_results[scenario_name] = None

### Multi-Turn Simulations — Scenarios 1 & 3: Direct Override and Goal Drift

In [ ]:
for scenario_name, driver_name in MULTI_TURN_MAP.items():
    if scenario_name not in registered_scenarios:
        print(f"⚠ Scenario {scenario_name} not found — skipping")
        continue
    if driver_name not in registered_drivers:
        print(f"⚠ Driver {driver_name} not found — skipping {scenario_name}")
        continue

    print(f"\n{'='*60}")
    print(f"Running simulation: {scenario_name}")
    print(f"Driver: {driver_name} | Max turns: {MAX_TURNS}")
    print(f"{'='*60}")

    try:
        driver_reg = registered_drivers[driver_name]
        multi_driver = Driver(
            temperature=driver_reg.temperature if hasattr(driver_reg, "temperature") else 0.7,
            name=driver_name,
            prompt_template=driver_reg.prompt_template,
        )
        run = okareo.run_simulation(
            target=target,
            driver=multi_driver,
            name=f"ASI01 Simulation — {scenario_name}",
            api_key=OKAREO_API_KEY,
            first_turn="target",
            scenario=registered_scenarios[scenario_name],
            max_turns=MAX_TURNS,
            checks=[GOAL_HIJACK_CHECK, GOAL_CONSISTENCY_CHECK],
        )
        all_results[scenario_name] = run
        print(f"  ✓ Simulation complete: {run.id}")
        if hasattr(run, "app_link") and run.app_link:
            print(f"  View: {run.app_link}")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        all_results[scenario_name] = None

### Results Summary

In [ ]:
print("\n" + "=" * 60)
print("ASI01 AGENT GOAL HIJACK — EVALUATION RESULTS")
print("OWASP Category: ASI01 | Risk Severity: Critical")
print("=" * 60)

print(f"\n{'Scenario':<46} {'Status':<10} {'Link / Run ID'}")
print("-" * 110)
for name, result in all_results.items():
    if result is None:
        print(f"{name:<46} {'ERROR':<10} N/A")
    else:
        link = getattr(result, "app_link", None) or result.id
        print(f"{name:<46} {'COMPLETE':<10} {link}")

errors = sum(1 for r in all_results.values() if r is None)
print(f"\nTotal evaluated: {len(all_results)} | Errors: {errors}")
if not errors:
    print("✓ All scenarios completed. See Okareo dashboard for full results.")

### Detailed Results (Optional)

Retrieve per-row scores and conversation transcripts for any completed simulation run.

In [ ]:
# Uncomment to inspect a specific completed run in detail:
# RUN_ID = "<paste_run_id_here>"
# detailed = okareo.get_test_run(RUN_ID)
# for row in (detailed.model_results or []):
#     print(f"Input:   {str(row.get('scenario_input', ''))[:80]}")
#     print(f"Output:  {str(row.get('model_output', ''))[:80]}")
#     print(f"Checks:  {row.get('check_scores', {})}")
#     print("-" * 40)